# 🤖 Fluxos Inteligentes com Python + IA
## Etapa 3 — Classificação Inteligente com Gemini API

> **Objetivo:** Enviar o texto extraído na Etapa 2 para o Gemini, que vai classificar cada documento e extrair as informações-chave de forma estruturada (JSON).

---
**Bibliotecas desta etapa:** `google-generativeai`, `json`, `pathlib`

**Modelo usado:** `gemini-2.0-flash` (padrão) com fallback automático para outros modelos disponíveis.

## 📦 1. Instalação das dependências

In [56]:
!pip install google-generativeai --quiet
print('✅ google-generativeai instalado!')

✅ google-generativeai instalado!


## 🔑 2. Configurando a API Key e o cliente Gemini

> **Segurança:** Nunca cole sua chave direto no código! Usamos `userdata` do Colab para guardar com segurança.
>
> Para adicionar: no menu lateral do Colab clique em 🔑 **Secrets** → **Add new secret** → nome: `GEMINI_API_KEY` → cole sua chave.

In [57]:
import google.generativeai as genai
import json
import time
from datetime import datetime
from pathlib import Path
from google.colab import userdata

genai.configure(api_key=userdata.get("GEMINI_API_KEY"))

# Ordem de preferência: mais rápidos/baratos primeiro, pesados como fallback
MODELOS = [
    "gemini-2.0-flash-lite",       # rápido e gratuito
    "gemini-2.0-flash-lite-001",
    "gemini-2.5-flash-lite",
    "gemini-2.0-flash",            # médio
    "gemini-2.0-flash-001",
    "gemini-2.5-flash",
    "gemini-2.5-pro",              # pesado (última opção)
]

ERROS_TRANSITORIOS = (
    "429", "quota", "resource_exhausted",
    "connection aborted", "remote end closed", "connection reset",
)

def gerar_conteudo(prompt: str, json_mode: bool = False) -> str:
    """
    Tenta gerar conteúdo percorrendo MODELOS em ordem.
    Erros transitórios (quota/rede) passam para o próximo modelo.
    Bloqueia e tenta novamente quando todos estão indisponíveis.
    """
    generation_config = {"response_mime_type": "application/json"} if json_mode else {}

    while True:
        for nome in MODELOS:
            try:
                modelo = genai.GenerativeModel(model_name=nome)
                resposta = modelo.generate_content(prompt, generation_config=generation_config)
                print(f"      ✓ {nome}")
                return resposta.text
            except Exception as e:
                erro = str(e).lower()
                if any(t in erro for t in ERROS_TRANSITORIOS):
                    print(f"      ⏭ quota/rede → {nome}")
                    continue
                print(f"      ✗ erro fatal → {nome}: {e}")
                continue  # tenta o próximo mesmo em erros não-transitórios

        print("      ⏳ Todos os modelos indisponíveis. Aguardando 30s...")
        time.sleep(30)

# Teste rápido de conectividade
print(gerar_conteudo("Responda apenas: OK"))

      ⏭ quota/rede → gemini-2.0-flash-lite


      ⏭ quota/rede → gemini-2.0-flash-lite-001
      ✓ gemini-2.5-flash-lite
OK


## 📂 3. Carregando os JSONs da Etapa 2

> ⚠️ **Pré-requisito:** Execute os notebooks das **Etapas 1 e 2** antes deste.

In [58]:
PASTA_RESULTADOS = Path('resultados')
PASTA_RESULTADOS.mkdir(exist_ok=True)

arquivo_consolidado = PASTA_RESULTADOS / 'extracao_consolidada.json'

with open(arquivo_consolidado, 'r', encoding='utf-8') as f:
    consolidado = json.load(f)

documentos = consolidado['documentos']

print(f'📂 Arquivo carregado : {arquivo_consolidado}')
print(f'📄 Documentos prontos: {len(documentos)}\n')

for doc in documentos:
    print(f'  ✔  {doc["arquivo"]:<45} tipo: {doc["tipo_doc"]}')

📂 Arquivo carregado : resultados/extracao_consolidada.json
📄 Documentos prontos: 7

  ✔  NF_001_ComercioNordeste.pdf                   tipo: nota_fiscal
  ✔  NF_002_IndustriaPernambucana.pdf              tipo: nota_fiscal
  ✔  OS_047_ConsultoriaAlpha.pdf                   tipo: ordem_servico
  ✔  OS_048_SupermercadoBomPreco.pdf               tipo: ordem_servico
  ✔  OS_049_ClinicaSaudeTotal.pdf                  tipo: ordem_servico
  ✔  Planilha_Estoque_Maio2025.pdf                 tipo: planilha_estoque
  ✔  Relatorio_Abril_2025.pdf                      tipo: relatorio


## 🧠 4. O Prompt — instruindo o Gemini

A chave de um bom pipeline de IA está no **prompt**. Aqui usamos **structured output prompting**: pedimos ao modelo que responda **exclusivamente em JSON**, com campos definidos por nós.

Isso garante que a saída sempre seja parseável e previsível.

In [59]:
def montar_prompt(texto: str, tipo_doc: str, arquivo: str) -> str:
    """Monta o prompt de classificação estruturada para o Gemini."""
    return f"""Você é um especialista em análise e classificação de documentos financeiros e jurídicos.

Analise o documento abaixo e responda EXCLUSIVAMENTE em JSON válido, sem texto adicional.

DOCUMENTO
  Arquivo  : {arquivo}
  Tipo hint: {tipo_doc}
  Conteúdo :
{texto[:4000]}

FORMATO DE RESPOSTA (JSON):
{{
  "tipo_documento": "<categoria precisa, ex: Contrato de Prestação de Serviços>",
  "confianca": "<alta | media | baixa>",
  "status": "<ativo | encerrado | pendente | indefinido>",
  "resumo": "<resumo objetivo em 1-2 frases>",
  "entidades": {{
    "parte_1": "<nome da primeira parte>",
    "parte_2": "<nome da segunda parte>",
    "responsavel": "<nome do responsável, se houver>"
  }},
  "valores": {{
    "principal": "<valor principal, ex: R$ 10.000,00>",
    "multa": "<valor de multa, se houver>",
    "outros": "<outros valores relevantes>"
  }},
  "datas": {{
    "inicio": "<data de início no formato DD/MM/AAAA>",
    "vencimento": "<data de vencimento ou fim>",
    "assinatura": "<data de assinatura>"
  }},
  "indicadores": ["<item relevante 1>", "<item relevante 2>"],
  "itens_criticos": ["<cláusula ou risco crítico 1>", "<cláusula ou risco crítico 2>"]
}}
"""

# Teste de sanidade — garante que a função está disponível
print("✅ montar_prompt definida.")
print("   Exemplo de chamada:")
print(f"   montar_prompt(texto='...', tipo_doc='pdf', arquivo='exemplo.pdf')")

✅ montar_prompt definida.
   Exemplo de chamada:
   montar_prompt(texto='...', tipo_doc='pdf', arquivo='exemplo.pdf')


## 🔬 5. Teste com um único documento

Antes de processar tudo, validamos com o primeiro documento para garantir que o Gemini responde corretamente.

In [60]:
def classificar_documento(doc: dict) -> dict:
    """
    Envia um documento ao Gemini (com fallback) e retorna o resultado classificado.
    Em caso de JSON inválido, retorna um resultado de fallback com a resposta bruta.
    """
    prompt = montar_prompt(doc['texto'], doc['tipo_doc'], doc['arquivo'])
    texto  = gerar_conteudo(prompt, json_mode=True).strip()

    try:
        classificacao = json.loads(texto)
        status_parse  = 'ok'
    except json.JSONDecodeError:
        classificacao = {'erro': 'JSON inválido', 'resposta_bruta': texto}
        status_parse  = 'erro_parse'

    return {
        'arquivo':         doc['arquivo'],
        'tipo_original':   doc['tipo_doc'],
        'classificado_em': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'status_parse':    status_parse,
        'classificacao':   classificacao,
    }


# Testa com o primeiro documento
print(f"🔬 Testando: {documentos[0]['arquivo']}\n")
resultado_teste = classificar_documento(documentos[0])

print(f"\n  status_parse : {resultado_teste['status_parse']}")
print(f"  tipo         : {resultado_teste['classificacao'].get('tipo_documento', '?')}")
print(f"  confiança    : {resultado_teste['classificacao'].get('confianca', '?')}")
print(f"  resumo       : {resultado_teste['classificacao'].get('resumo', '')[:100]}")

🔬 Testando: NF_001_ComercioNordeste.pdf



      ⏭ quota/rede → gemini-2.0-flash-lite


      ⏭ quota/rede → gemini-2.0-flash-lite-001
      ✓ gemini-2.5-flash-lite

  status_parse : ok
  tipo         : Nota Fiscal Eletrônica
  confiança    : alta
  resumo       : Nota fiscal eletrônica emitida pela Tech Solutions Ltda para a Comercio Nordeste SA, referente à ven


## 🚀 6. Classificando todos os documentos

In [61]:
todos_classificados = []
erros_classificacao  = []

print(f'🤖 Classificando {len(documentos)} documentos com Gemini...\n')

for i, doc in enumerate(documentos, start=1):
    print(f'  [{i}/{len(documentos)}] {doc["arquivo"]}')
    try:
        resultado = classificar_documento(doc)
        todos_classificados.append(resultado)

        cl     = resultado['classificacao']
        tipo   = cl.get('tipo_documento', '?')
        conf   = cl.get('confianca', '?')
        resumo = cl.get('resumo', '')[:70]
        print(f'      tipo: {tipo:<25} confiança: {conf:<6} | {resumo}...')

    except Exception as e:
        erros_classificacao.append({'arquivo': doc['arquivo'], 'erro': str(e)})
        print(f'      ❌ ERRO: {e}')

    time.sleep(1)  # respeita rate limit gratuito

print(f'\n✅ Classificados : {len(todos_classificados)}')
print(f'❌ Erros         : {len(erros_classificacao)}')

🤖 Classificando 7 documentos com Gemini...

  [1/7] NF_001_ComercioNordeste.pdf


      ⏭ quota/rede → gemini-2.0-flash-lite


      ⏭ quota/rede → gemini-2.0-flash-lite-001
      ✓ gemini-2.5-flash-lite
      tipo: Nota Fiscal Eletrônica    confiança: alta   | Nota Fiscal Eletrônica emitida pela Tech Solutions Ltda para a Comerci...
  [2/7] NF_002_IndustriaPernambucana.pdf


      ⏭ quota/rede → gemini-2.0-flash-lite


      ⏭ quota/rede → gemini-2.0-flash-lite-001
      ✓ gemini-2.5-flash-lite
      tipo: Nota Fiscal               confiança: alta   | Nota Fiscal emitida pela Tech Solutions Ltda para a Industria Pernambu...
  [3/7] OS_047_ConsultoriaAlpha.pdf


...

      ⏭ quota/rede → gemini-2.0-flash-lite-001
      ✓ gemini-2.5-flash-lite
      tipo: Ordem de Serviço          confiança: alta   | Ordem de serviço para instalação de software hospitalar, migração de b...
  [6/7] Planilha_Estoque_Maio2025.pdf


      ⏭ quota/rede → gemini-2.0-flash-lite


      ⏭ quota/rede → gemini-2.0-flash-lite-001
      ✓ gemini-2.5-flash-lite
      tipo: Controle de Estoque       confiança: alta   | Relatório de controle de estoque da Tech Solutions Ltda para maio de 2...
  [7/7] Relatorio_Abril_2025.pdf


      ⏭ quota/rede → gemini-2.0-flash-lite


      ⏭ quota/rede → gemini-2.0-flash-lite-001
      ✓ gemini-2.5-flash-lite
      tipo: Relatório Operacional     confiança: alta   | Relatório operacional referente a Abril de 2025 para a Tech Solutions ...

✅ Classificados : 7
❌ Erros         : 0


## 💾 7. Salvando os resultados

In [62]:
modelo_usado = MODELOS[0]  # referência ao primeiro da lista (mais usado)

# Salva um JSON por documento
for resultado in todos_classificados:
    nome_base = Path(resultado['arquivo']).stem
    caminho   = PASTA_RESULTADOS / f'{nome_base}_classificado.json'
    with open(caminho, 'w', encoding='utf-8') as f:
        json.dump(resultado, f, ensure_ascii=False, indent=2)
    print(f'  💾 {caminho}')

# Salva o consolidado final
consolidado_final = {
    'total_documentos': len(todos_classificados),
    'erros':            erros_classificacao,
    'modelos_tentados': MODELOS,
    'gerado_em':        datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'documentos':       todos_classificados,
}

with open(PASTA_RESULTADOS / 'classificacao_final.json', 'w', encoding='utf-8') as f:
    json.dump(consolidado_final, f, ensure_ascii=False, indent=2)

print(f'\n📦 Consolidado salvo: resultados/classificacao_final.json')

  💾 resultados/NF_001_ComercioNordeste_classificado.json
  💾 resultados/NF_002_IndustriaPernambucana_classificado.json
  💾 resultados/OS_047_ConsultoriaAlpha_classificado.json
  💾 resultados/OS_048_SupermercadoBomPreco_classificado.json
  💾 resultados/OS_049_ClinicaSaudeTotal_classificado.json
  💾 resultados/Planilha_Estoque_Maio2025_classificado.json
  💾 resultados/Relatorio_Abril_2025_classificado.json

📦 Consolidado salvo: resultados/classificacao_final.json


## 🔍 8. Inspecionando a classificação completa de um documento

In [63]:
INDICE = 0  # mude para inspecionar outro documento

if not todos_classificados:
    print("❌ A lista está vazia — verifique se a Etapa 6 rodou com sucesso.")
else:
    try:
        r  = todos_classificados[INDICE]
        cl = r['classificacao']

        print('=' * 60)
        print(f'  ARQUIVO    : {r["arquivo"]}')
        print(f'  TIPO       : {cl.get("tipo_documento")}')
        print(f'  CONFIANÇA  : {cl.get("confianca")}')
        print(f'  STATUS     : {cl.get("status")}')
        print('=' * 60)

        print(f'\n📝 Resumo:\n   {cl.get("resumo")}')

        print('\n🏢 Entidades:')
        for k, v in cl.get('entidades', {}).items():
            print(f'   {k:<25}: {v}')

        print('\n💰 Valores:')
        for k, v in cl.get('valores', {}).items():
            print(f'   {k:<25}: {v}')

        print('\n📅 Datas:')
        for k, v in cl.get('datas', {}).items():
            print(f'   {k:<25}: {v}')

        if cl.get('indicadores'):
            print('\n📊 Indicadores encontrados:')
            for ind in cl['indicadores']:
                print(f'   • {ind}')

        if cl.get('itens_criticos'):
            print('\n⚠️  Itens críticos:')
            for item in cl['itens_criticos']:
                print(f'   • {item}')

    except IndexError:
        print(f"❌ Índice {INDICE} não existe (total: {len(todos_classificados)}).")

  ARQUIVO    : NF_001_ComercioNordeste.pdf
  TIPO       : Nota Fiscal Eletrônica
  CONFIANÇA  : alta
  STATUS     : indefinido

📝 Resumo:
   Nota Fiscal Eletrônica emitida pela Tech Solutions Ltda para a Comercio Nordeste SA, referente à venda de equipamentos de informática.

🏢 Entidades:
   parte_1                  : Tech Solutions Ltda
   parte_2                  : Comercio Nordeste SA
   responsavel              : None

💰 Valores:
   principal                : R$ 11.643,20
   multa                    : None
   outros                   : None

📅 Datas:
   inicio                   : None
   vencimento               : None
   assinatura               : None

📊 Indicadores encontrados:
   • CNPJ: 12.345.678/0001-90
   • CNPJ: 98.765.432/0001-11
   • Local de emissão: Recife, PE


## 📊 9. Resumo da Etapa 3

In [64]:
ok     = [r for r in todos_classificados if r['status_parse'] == 'ok']
falhos = [r for r in todos_classificados if r['status_parse'] != 'ok']

confs: dict[str, int] = {}
for r in ok:
    nivel = r['classificacao'].get('confianca', 'desconhecida')
    confs[nivel] = confs.get(nivel, 0) + 1

print('=' * 55)
print('  ETAPA 3 CONCLUÍDA — Classificação com Gemini')
print('=' * 55)
print(f'  Documentos classificados : {len(todos_classificados)}')
print(f'  JSON válidos             : {len(ok)}')
print(f'  Erros de parse           : {len(falhos)}')
print(f'  Modelos disponíveis      : {len(MODELOS)}')
print()
print('  Nível de confiança:')
for nivel, qtd in confs.items():
    print(f'    • {nivel:<12} {qtd} doc(s)')
print()
print('  Arquivos gerados:')
print('    • resultados/<nome>_classificado.json')
print('    • resultados/classificacao_final.json')
print('=' * 55)
print()
print('  Próximo passo: Etapa 4 — Geração do Relatório Final')
print('=' * 55)

  ETAPA 3 CONCLUÍDA — Classificação com Gemini
  Documentos classificados : 7
  JSON válidos             : 7
  Erros de parse           : 0
  Modelos disponíveis      : 7

  Nível de confiança:
    • alta         7 doc(s)

  Arquivos gerados:
    • resultados/<nome>_classificado.json
    • resultados/classificacao_final.json

  Próximo passo: Etapa 4 — Geração do Relatório Final
